# PneumoNet — ResNet50 Transfer Learning
Two-stage fine-tuning of pretrained ResNet50 for pneumonia detection.

**Stage 1:** Train the classification head only (frozen backbone, 12 epochs).  
**Stage 2:** Unfreeze all layers and fine-tune end-to-end (5 epochs).

**Setup (run once):**
```bash
pip install -r ../requirements.txt
```

In [ ]:
import sys
sys.path.insert(0, '..')

import yaml
import torch
import torch.optim as optim
from src.data_loader import get_dataloaders
from src.models import build_resnet50, load_model
from src.train import train_loop, get_weighted_criterion
from src.evaluate import evaluate, plot_confusion_matrix, plot_roc_curve, plot_training_curves
from src.explain import show_gradcam, show_shap, show_lime

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Load Dataset

In [ ]:
train_loader, val_loader, test_loader, train_data, val_data, test_data = get_dataloaders(
    dataset_path=cfg['dataset']['path'],
    batch_size=cfg['dataset']['batch_size'],
    num_workers=cfg['dataset']['num_workers'],
    image_size=cfg['dataset']['image_size'],
)
print(f'Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')
print('Classes:', train_data.classes)

## 2. Build ResNet50 (frozen backbone)

In [ ]:
model = build_resnet50(freeze_backbone=True).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable:,} (head only)')

## 3. Stage 1 — Head Training (frozen backbone)

In [ ]:
cfg_r50   = cfg['training']['resnet50']
criterion = get_weighted_criterion(train_data, device)
optimizer = optim.Adam(model.fc.parameters(), lr=cfg_r50['head_lr'])
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=cfg_r50['scheduler_step'],
                                       gamma=cfg_r50['scheduler_gamma'])

history1 = train_loop(
    model, train_loader, val_loader, criterion, optimizer,
    num_epochs=cfg_r50['head_epochs'],
    device=device,
    save_path=cfg['models']['resnet50_path'],
    scheduler=scheduler,
)

## 4. Stage 2 — Fine-Tune All Layers

In [ ]:
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=cfg_r50['finetune_lr'])

history2 = train_loop(
    model, train_loader, val_loader, criterion, optimizer,
    num_epochs=cfg_r50['finetune_epochs'],
    device=device,
    save_path=cfg['models']['resnet50_path'],
)

import torch
torch.save(model.state_dict(), cfg['models']['resnet50_path'])
print('Final model saved.')

## 5. Training Curves

In [ ]:
plot_training_curves(history1, title_suffix='(Head Training)')
plot_training_curves(history2, title_suffix='(Fine-Tuning)')

## 6. Evaluate on Test Set

In [ ]:
model = load_model(build_resnet50(freeze_backbone=False), cfg['models']['resnet50_path'], device)
metrics, y_true, y_pred, y_probs = evaluate(model, test_loader, device)
plot_confusion_matrix(y_true, y_pred, classes=test_data.classes)
plot_roc_curve(y_true, y_probs, metrics['roc_auc'])

## 7. Explainability

In [ ]:
sample_img, _ = test_data[0]
target_layer  = model.layer4[-1].conv3  # last conv in ResNet50
show_gradcam(model, sample_img, target_layer, device)

In [ ]:
import torch
test_imgs = torch.stack([test_data[i][0] for i in range(8)])
show_shap(model, test_imgs, device)

In [ ]:
sample_img, _ = test_data[0]
show_lime(model, sample_img.unsqueeze(0), device)

## 8. Gradio Demo
Or from terminal: `python -m src.app --model resnet50 --weights models/pneumonet_resnet50_finetuned.pth`

In [ ]:
from src.app import load_resnet, launch_app

app_model = load_resnet(cfg['models']['resnet50_path'])
launch_app(app_model, title='PneumoNet ResNet50 - Pneumonia Detection')